# Fine-tune a VLM on scene-text Q&A — with LanceDB, on a free Colab T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lancedb/training/blob/vlm-textvqa/examples/vlm-textvqa/notebooks/colab_textvqa_lance.ipynb)

This notebook runs the **whole VLM fine-tuning loop end-to-end on a single free T4** — the same pipeline that runs at scale on an H100 in [`examples/vlm-textvqa`](https://github.com/lancedb/training/tree/vlm-textvqa/examples/vlm-textvqa), shrunk to a Colab-sized subset.

**The model:** `Qwen2.5-VL-3B-Instruct`, LoRA-tuned for [TextVQA](https://textvqa.org) (read the text *in* an image, answer a question about it).

**Why it fits a 16 GB T4 — the LanceDB trick:** the vision tower is the expensive part of a VLM. We run it **once**, offline, and store its output (`vision_tower_hiddens`) as a column in a Lance table. Training then reads that column straight off disk and skips the vision tower entirely — so the train loop only holds the (4-bit quantized) language model + a LoRA adapter. This is `Curate → Manage → Load & train` from the [repo README](https://github.com/lancedb/training#why-use-lancedb-for-training), at toy scale.

What you'll run:
1. **Download** a pre-baked Lance subset (cached vision hiddens already computed on a GPU).
2. **Benchmark** read throughput: the same cached columns from **Lance vs Parquet**.
3. **QLoRA fine-tune** from the cached columns — vision tower never loaded.
4. **Before / after**: generate answers with the base model and the tuned model, side by side.

> ⏱️ End-to-end on a T4: a few minutes. This is a *demo-scale* run (hundreds of rows, tens of steps) — the point is the mechanics and the throughput story, not a SOTA checkpoint.

## 0 · Check the GPU

Runtime → Change runtime type → **T4 GPU**. The cell below should print a T4 (or better).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime → Change runtime type → T4 GPU'
print('torch', torch.__version__, '| device', torch.cuda.get_device_name(0))

## 1 · Setup

Clone the repo and install just what the Colab path needs. Colab already ships a CUDA-enabled torch, so we don't reinstall it — we add the data + model libraries and put the `vlm` package on the path (no full `pip install -e .`, which would drag in the GPU-only Geneva/WebDataset stack).

In [ ]:
import os, sys
REPO_DIR = '/content/training'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch vlm-textvqa https://github.com/lancedb/training.git {REPO_DIR}
EXAMPLE_DIR = f'{REPO_DIR}/examples/vlm-textvqa'
sys.path.insert(0, EXAMPLE_DIR)   # make `import vlm.*` work without installing the package
%cd {EXAMPLE_DIR}

In [ ]:
# Targeted installs (quiet). torch/numpy/pillow/pyarrow already exist on Colab.
!pip -q install 'lancedb>=0.30' 'pylance>=0.18' 'transformers>=4.49' 'peft>=0.13' 'accelerate>=1.0' \
    'bitsandbytes>=0.43' 'qwen-vl-utils>=0.0.8' 'huggingface_hub>=0.24' 2>/dev/null
print('deps installed')

## 2 · Download the pre-baked Lance subset

The fast path reads `vision_tower_hiddens` — and computing those needs a GPU pass over the images. We did that once with [`vlm/colab_prepare.py`](https://github.com/lancedb/training/blob/vlm-textvqa/examples/vlm-textvqa/vlm/colab_prepare.py) and hosted the result, so this notebook just downloads it:

- `textvqa_colab_train.lance` — train subset **with** cached Tier-3 columns
- `textvqa_colab_val.lance` — eval subset (raw images, for before/after)
- `cached_train.parquet` — the same cached columns mirrored to Parquet, for the throughput cell

> If you'd rather bake your own (or the hosted repo isn't up yet), run on any GPU box:
> `python -m vlm.colab_prepare --out data/colab --train-rows 512 --val-rows 64 --hf-repo <your-org>/textvqa-lance-colab --push`

In [ ]:
from huggingface_hub import snapshot_download

# Where the baked subset lives. Override if you hosted your own.
HF_REPO = os.environ.get('TEXTVQA_COLAB_REPO', 'lance-format/textvqa-lance-colab')

local = snapshot_download(repo_id=HF_REPO, repo_type='dataset', local_dir='data/colab')
TRAIN_LANCE = f'{local}/textvqa_colab_train.lance'
VAL_LANCE   = f'{local}/textvqa_colab_val.lance'
PARQUET     = f'{local}/cached_train.parquet'

# The local tables are LanceDB tables at <uri>/<name>.lance.
import lancedb, os
def open_tbl(path):
    name = os.path.basename(path)
    name = name[:-len('.lance')] if name.endswith('.lance') else name
    return lancedb.connect(os.path.dirname(path)).open_table(name)

train_tbl, val_tbl = open_tbl(TRAIN_LANCE), open_tbl(VAL_LANCE)
print('train rows:', train_tbl.count_rows())
print('val rows:  ', val_tbl.count_rows())
print('cached columns:', [c for c in train_tbl.schema.names if c in
      ('vision_tower_hiddens','input_ids','attention_mask','labels','sft_tokens')])

## 3 · Throughput: the cached columns, Lance vs Parquet

This is the read your train loop actually does: **shuffled, batched random access** over the cached feature columns. We time it against the *same bytes* stored two ways.

- **Lance** does random `take(indices)` straight off disk — it never has to hold the whole table in RAM, so this is exactly what scales to the full 34k-row / ~57 GB cached corpus on the same box (or streamed from object storage).
- **Parquet** has no row-level random access; the realistic loader pattern is to read the file into memory once, then gather. Fine at toy scale, but it's loading the whole thing into RAM to do it.

Numbers are printed live from *your* T4 — we don't hard-code them.

In [ ]:
import time, numpy as np, pyarrow as pa, pyarrow.parquet as pq
from lancedb.permutation import Permutation

# Cached feature columns (flat from the direct bake; struct if baked via Geneva).
names = set(train_tbl.schema.names)
if "sft_tokens" in names and "input_ids" not in names:
    CACHED = ["vision_tower_hiddens", "sft_tokens"]
else:
    CACHED = ["vision_tower_hiddens", "input_ids", "attention_mask", "labels"]
BATCH, N_BATCHES = 8, 40

n = train_tbl.count_rows()
rng = np.random.default_rng(0)
index_batches = [rng.permutation(n)[:BATCH].tolist() for _ in range(N_BATCHES)]

def _touch(obj):
    # materialize the heavy column the way a collate fn would
    col = obj.column("vision_tower_hiddens")
    if isinstance(col, pa.ChunkedArray):
        col = col.combine_chunks()
    col.values.to_numpy(zero_copy_only=False)

# --- Lance: random access off disk via the LanceDB Permutation API ---
perm = Permutation.identity(train_tbl).select_columns(CACHED).with_format("arrow")
_touch(perm.__getitems__(index_batches[0]))  # warmup
t0 = time.time()
for idx in index_batches:
    _touch(perm.__getitems__(idx))
lance_s = time.time() - t0

# --- Parquet: load the whole file into RAM once, then gather (the usual pattern) ---
t0 = time.time()
ptbl = pq.read_table(PARQUET, columns=CACHED)   # whole file into RAM
for idx in index_batches:
    _touch(ptbl.take(pa.array(idx)))
pq_s = time.time() - t0

samples = BATCH * N_BATCHES
print(f"Lance   : {samples/lance_s:7.1f} samples/s  ({lance_s:.2f}s, Permutation random access off disk)")
print(f"Parquet : {samples/pq_s:7.1f} samples/s  ({pq_s:.2f}s, incl. full-file load into RAM)")
print()
print("Takeaway: at this size both are fast. The point is the Permutation API gives you")
print("shuffled random access WITHOUT loading the corpus into RAM — which is what lets the")
print("same loop scale to the full 57 GB cached table, local or from object storage.")

## 4 · QLoRA fine-tune — from the cached columns

We reuse the repo's own training building blocks (`_build_model`, `_forward_cached`, `make_cached_loader`) so this is the *real* code path, just driven inline so you can watch it.

`_build_model(..., load_4bit=True)`:
- loads the LLM in **4-bit NF4** (bitsandbytes) — ~2 GB instead of ~7.5 GB,
- **deletes the vision tower** (we have its output cached), and
- wraps the LLM's q/k/v/o with a LoRA adapter.

The loop pulls `vision_tower_hiddens` + `input_ids` + `labels` from Lance and injects the cached hiddens at the `<|image_pad|>` positions via `masked_scatter`. No vision tower, no image decode, no tokenization in the loop.

In [ ]:
from transformers import AutoTokenizer
from vlm.train_qwen25vl_lora import _build_model, _forward_cached, _QWEN_MODEL_ID, _IMAGE_PAD_TOKEN
from vlm.dataloader import make_cached_loader

tok = AutoTokenizer.from_pretrained(_QWEN_MODEL_ID)
image_pad_id = tok.convert_tokens_to_ids(_IMAGE_PAD_TOKEN)

model = _build_model(use_lora=True, lora_r=16, load_4bit=True)
model.train()

trainable = [p for p in model.parameters() if p.requires_grad]
optim = torch.optim.AdamW(trainable, lr=2e-4, betas=(0.9, 0.95))
device = torch.device("cuda:0")

In [ ]:
MAX_STEPS = 40            # demo scale; bump for a stronger adapter
GRAD_ACCUM = 4
loader = make_cached_loader(TRAIN_LANCE, batch_size=2, num_workers=0, shuffle=True, seed=0)

step, accum, t0 = 0, 0, time.time()
optim.zero_grad(set_to_none=True)
for batch in loader:
    batch = batch.to(device)
    loss = _forward_cached(model, batch, image_pad_id)
    (loss / GRAD_ACCUM).backward()
    accum += 1
    if accum >= GRAD_ACCUM:
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optim.step(); optim.zero_grad(set_to_none=True)
        accum = 0; step += 1
        sps = (step * GRAD_ACCUM * 2) / (time.time() - t0)
        if step % 5 == 0 or step == MAX_STEPS:
            print(f'step {step:3d}/{MAX_STEPS}  loss={loss.item():.4f}  {sps:.1f} samples/s')
        if step >= MAX_STEPS:
            break

ADAPTER_DIR = 'runs/colab_lora/lora'
model.save_pretrained(ADAPTER_DIR)
print('saved adapter to', ADAPTER_DIR)

In [ ]:
# free the training model before loading the full model for eval
del model, optim, loader
import gc; gc.collect(); torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 5 · Before / after — does the tuned model read text better?

Now we load the **full** model (vision tower included, in 4-bit) and generate on a handful of *held-out* val images — once with the base weights, once with our LoRA adapter — and show them side by side. Reuses `vlm.eval`'s generation + scoring + thumbnail helpers.

In [ ]:
import io
from PIL import Image
from vlm.eval import _load_model, _generate, _score_one, _b64_thumb

K = 6
rows = val_tbl.search().select(['image','question','answer','answers']).limit(K).to_arrow().to_pylist()

def run(adapter):
    m, proc = _load_model(adapter_dir=adapter, load_4bit=True)
    outs = []
    for r in rows:
        img = Image.open(io.BytesIO(r['image'])).convert('RGB')
        outs.append(_generate(m, proc, img, r['question']))
    del m; gc.collect(); torch.cuda.empty_cache()
    return outs

base_ans  = run(None)
tuned_ans = run(ADAPTER_DIR)

In [ ]:
from IPython.display import HTML, display

head = ('<tr><th>Image</th><th>Question</th><th>Base</th>'
        '<th>Tuned</th><th>Ground truth</th></tr>')
trs = []
base_score = tuned_score = 0.0
for r, b, t in zip(rows, base_ans, tuned_ans):
    gts = r['answers'][:5]
    bs, ts = _score_one(b, r['answers']), _score_one(t, r['answers'])
    base_score += bs; tuned_score += ts
    win = 'style="background:#e6ffe6"' if ts > bs else ''
    thumb = _b64_thumb(r['image'])
    trs.append(
        f'<tr {win}><td><img src="data:image/jpeg;base64,{thumb}" width=160/></td>'
        f'<td>{r["question"]}</td><td>{b}</td><td><b>{t}</b></td>'
        f'<td>{", ".join(gts)}</td></tr>')

display(HTML(f'<table>{head}{"".join(trs)}</table>'))
print(f'base  accuracy on these {K}: {base_score/K:.3f}')
print(f'tuned accuracy on these {K}: {tuned_score/K:.3f}')
print('(green rows = tuned beat base. K is tiny here — run the full pipeline for real numbers.)')

## Recap

On a free T4 you just ran the full shape of a real VLM fine-tune:

| Stage | What LanceDB did |
|---|---|
| **Curate / Manage** | the expensive vision-tower output lives as a column you computed once and reuse forever (`vision_tower_hiddens`) |
| **Load & train** | shuffled random access off disk — no full-corpus load into RAM — feeding a vision-tower-free, 4-bit train loop |
| **Eval** | base vs tuned, side by side, on held-out images |

**Scale it up:** the same code runs the full 34,602-row corpus on an H100 — see [`examples/vlm-textvqa`](https://github.com/lancedb/training/tree/vlm-textvqa/examples/vlm-textvqa). There the cached path trains at **~2× the throughput** of running the vision tower every step, at **−1.3 GB VRAM**, and lifts TextVQA val accuracy **0.793 → 0.815**.